<a href="https://colab.research.google.com/github/faculatini/flyrank_ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faculatini/flyrank_ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [8]:
!git clone https://github.com/faculatini/flyrank_ml.git

Cloning into 'flyrank_ml'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 135 (delta 46), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 1.84 MiB | 14.94 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [9]:
%cd flyrank_ml

/content/flyrank_ml


In [11]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 30,000
Columns: 44


In [12]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nClients:", df["client_id"].nunique())
print("Content items:", df["content_id"].nunique())

print("\nDuplicate content_id rows:", df["content_id"].duplicated().sum())

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Clients: 32
Content items: 30000

Duplicate content_id rows: 0


In [13]:
signal_columns = [
    "days_since_last_update",
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "engagement_rate",
    "trend_direction",
    "trend_pct",
]

df[signal_columns].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
days_since_last_update,30000.0,NaN,NaN,NaN,46.0983,42.078709,1.0,20.0,20.0,104.0,373.0
content_age_days,30000.0,NaN,NaN,NaN,256.1678,132.70793,90.0,132.0,236.0,333.0,564.0
impressions_90d,30000.0,NaN,NaN,NaN,5200.3663,16838.019547,1.0,81.0,731.0,3615.25,517715.0
clicks_90d,30000.0,NaN,NaN,NaN,16.097333,75.076958,0.0,0.0,1.0,7.0,4178.0
ctr,30000.0,NaN,NaN,NaN,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,NaN,NaN,NaN,16.34238,15.21679,0.0,6.2,10.8,22.3,245.0
sessions_90d,30000.0,NaN,NaN,NaN,37.066633,107.069131,1.0,2.0,7.0,27.0,4345.0
engagement_rate,30000.0,NaN,NaN,NaN,2.53452,8.310096,0.0,0.0,0.0,1.35,100.0
trend_direction,30000,5,down,16262,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trend_pct,26612.0,NaN,NaN,NaN,-4.785969,473.86178,-100.0,-62.6,-33.5,0.0,44900.0


In [14]:
print("avg_position == 0:", (df["avg_position"] == 0).sum())
print("Missing ctr:", df["ctr"].isna().sum())
print("Missing impressions:", df["impressions_90d"].isna().sum())
print("Missing sessions:", df["sessions_90d"].isna().sum())
print("Missing days_since_last_update:", df["days_since_last_update"].isna().sum())

print("\nTrend direction:")
print(df["trend_direction"].value_counts(dropna=False))

avg_position == 0: 1205
Missing ctr: 0
Missing impressions: 0
Missing sessions: 0
Missing days_since_last_update: 0

Trend direction:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal checks and rule reasoning

Before encoding the baseline rule, I will check whether the signals behind the rule are actually visible in the data.

Each signal check uses a human-readable bucket table with:

* the signal bucket;
* the number of pages in the bucket (`n`);
* the observed outcome rate.

The verdict for each signal will be exactly one of:

* `CONFIRMED` — the expected pattern is clearly visible;
* `OPPOSITE` — the observed pattern points in the opposite direction;
* `MIXED` — the pattern appears only in part of the buckets or is not strong enough to stand alone;
* `FALSE` — the expected pattern is not visible.

A negative or mixed result is useful: it prevents a weak assumption from becoming part of the baseline rule.

At least one of the two checked signals must be connected to a real FlyRank flag discussed in the Week-4 session. The final rule will only use observable inputs available at decision time and will not use the decline label, `trend_direction`, or `trend_pct` as features.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.